## 1. Mount Google Drive
Ensure outputs are seamlessly backed up and available across all experiment runs.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

!mkdir -p /content/drive/MyDrive/afriqa_outputs


## 2. Install Dependencies

In [ ]:
import os

if not os.path.exists("afriqa-entity-aware-qa"):
    !git clone https://github.com/OmondiKevin/afriqa-entity-aware-qa.git

%cd afriqa-entity-aware-qa
!git pull
!pip install -e .
!pip install sentence-transformers sentencepiece protobuf peft

## 3. Configure Environment and Outputs
Symlink project outputs to Google Drive so all experiments persist to the same Drive-backed folder.

In [ ]:
import os
import torch
from transformers import set_seed

set_seed(42)

if os.path.exists("outputs"):
    !rm -rf outputs
!ln -s /content/drive/MyDrive/afriqa_outputs outputs


## 4. Download Dataset
Download original AfriQA and MasakhaNER JSONs.

In [ ]:
!python scripts/00_download_and_subset.py --config configs/default.yaml

## 5. Preprocess Dataset
Build baseline QA and multitask seq2seq-ready data once, then reuse across experiments.

In [ ]:
!python scripts/01_prepare_qa_data.py --config configs/default.yaml
!python scripts/01b_prepare_multitask_data.py --config configs/default.yaml

## 6. Confirm Default Model Config (`mT5-base`)
Confirm the default config uses `google/mt5-base` before running baseline and multitask mT5 experiments.

In [ ]:
!cat configs/default.yaml | grep -A 3 "model:"
!cat configs/default.yaml | grep -A 8 "run:"

## 7. Train Baseline QA (`mT5-base`)
Run the baseline QA experiment first.

In [ ]:
!python scripts/02_train_baseline_qa.py --config configs/default.yaml

## 8. Evaluate Baseline QA (`mT5-base`)
Evaluate baseline predictions and write baseline metrics.

In [ ]:
!python scripts/04_eval_predictions.py --config configs/default.yaml --pred_path outputs/predictions/baseline_mt5_test.jsonl

## 9. Train Multitask Sequential QA (`mT5-base`)
Confirm model/config and run the multitask sequential pipeline (NER -> QA) with `mT5-base`.

In [ ]:
!cat configs/default.yaml | grep -A 3 "model:"
!python scripts/03_train_multitask_qa.py --config configs/default.yaml --sequential

## 10. Evaluate Multitask Sequential QA (`mT5-base`)
Evaluate multitask predictions on QA-only rows.

In [ ]:
!python scripts/04_eval_predictions.py --config configs/default.yaml --pred_path outputs/predictions/multitask_mt5_test.jsonl --qa_only


## 11. Switch Model to ByT5 (`google/byt5-base`)
Create a derived Colab config so ByT5 outputs are written to distinct paths and do not overwrite mT5 results.

In [ ]:
import yaml
from pathlib import Path

base_cfg_path = Path("configs/default.yaml")
byt5_cfg_path = Path("configs/colab_byt5.yaml")

with open(base_cfg_path, "r", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)

cfg["model"]["base"] = "google/byt5-base"
cfg.setdefault("run", {})["multitask_output_dir"] = "outputs/checkpoints/multitask_byt5"
cfg["run"]["multitask_pred_path"] = "outputs/predictions/multitask_byt5_test.jsonl"

with open(byt5_cfg_path, "w", encoding="utf-8") as f:
    yaml.safe_dump(cfg, f, sort_keys=False, allow_unicode=False)

print(f"Wrote {byt5_cfg_path}")
!cat configs/colab_byt5.yaml | grep -A 3 "model:"
!cat configs/colab_byt5.yaml | grep -A 8 "run:"

## 12. Train Multitask Sequential QA (`ByT5-base`)
Run multitask sequential training using the derived ByT5 config.

In [ ]:
!cat configs/colab_byt5.yaml | grep -A 3 "model:"
!python scripts/03_train_multitask_qa.py --config configs/colab_byt5.yaml --sequential

## 13. Evaluate Multitask Sequential QA (`ByT5-base`)
Evaluate ByT5 multitask predictions on QA-only rows.

In [ ]:
!python scripts/04_eval_predictions.py --config configs/colab_byt5.yaml --pred_path outputs/predictions/multitask_byt5_test.jsonl --qa_only

## 14. Verify Saved Checkpoints and Outputs
Confirm that checkpoints and result files are present for each completed run.

In [ ]:
!ls -lh outputs/checkpoints
!ls -lh outputs/predictions
!ls -lh outputs/metrics

## 15. Display Final Metrics Summary (All Experiments)
Show overall metrics for baseline `mT5-base`, multitask sequential `mT5-base`, and multitask sequential `ByT5-base`.

In [ ]:
import json
import os

metrics_map = {
    "baseline_mt5": "outputs/metrics/baseline_mt5_test_metrics.json",
    "multitask_mt5_qa_only": "outputs/metrics/multitask_mt5_test_qa_only_metrics.json",
    "multitask_byt5_qa_only": "outputs/metrics/multitask_byt5_test_qa_only_metrics.json",
}

for name, path in metrics_map.items():
    print(f"\n=== {name} ===")
    if os.path.exists(path):
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
        print(json.dumps(data.get("overall", {}), indent=2, ensure_ascii=False))
    else:
        print(f"Missing metrics file: {path}")